# 1.生成式语言模型的对话模板介绍

在微调本地数据集的时候，模型名称的作用就是让对话模板可以根据模型名称自动匹配，这里的对话模板并不是官方的对话模板，而是参考官方的对话模板做的自己的对话模板。

![](Image/2025-04-05-16-45-46.png)

对话模板控制着输出内容的格式和信息，不同模型的对话模板是不一样的，但是LLaMAFactory里面同一个系列的模型一般是一样的，例如Qwen/Qwen2.5-1.5B-Instruct和Qwen/Qwen2.5-0.5B-Instruct的对话模板一致。但是Vllm推理框架没有自己的模板，用的是模型自带的模板，1.5B和2.5B模型自带的模板不一样，导致训练时候和部署时候使用的对话模板不一致，就会导致回答内容有很大差异。

![](Image/2025-04-05-16-59-00.png)

# 2.Lora微调后单独部署大模型输出结果不一致

不同的模型部署工具“对话模板”可能是不一样的，例如LLaMAFactory的对话模板和vllm、Ollama的对话模板不一样。模型使用过程一般遇到三个框架，一个是微调框架、模型推理框架、前端框架（例如OpenWebUI），这三个过程可能用的模板都不一样，导致微调测试和前端使用输出结果大相径庭

## 解决办法

解决微调训练框架(LLaMAFactory)和推理框架（vllm）模板不一直问题即----模板对齐

思路：
将LLaMAFactory微调训练时候的对话模板转换成推理框架（vllm）的.jinja模板格式，vllm加载转换后的模板

[vllm聊天模板](https://docs.vllm.com.cn/en/latest/serving/openai_compatible_server.html#chat-template)

![](Image/2025-04-06-00-29-13.png)

vllm serve <model> --chat-template ./path-to-chat-template.jinja

LLaMAFactory模板所在位置

/root/LLaMA-Factory/src/llamafactory/data/template.py

![](Image/2025-04-06-00-32-44.png)

LLaMAFactory本身就带模板格式转换函数，但是是私有函数，外面无法调用，我们可以写个脚本放到template.py同级目录下，然后调用该函数

![](Image/2025-04-06-00-42-41.png)

上面这个函数是私有的，下面这个是public的，可以调用

![](Image/2025-04-06-00-53-57.png)

![](Image/2025-04-06-00-49-53.png)

# 3.使用XTuner微调大模型

XTuner和LLama Factory使用方式上完全不同，LLaMA Factory是以可视化界面操作为主主流的框架XTuner恰恰相反，主要通过命令行和文件配置来微调模型，模型测试的方式也不一样，XTuner训练过程中以主观评价为主，LLaMA Factory在训练过程中是无法执行主观评价测试的，训练完成后在chat界面执行主观测试。

# 3.如何导出LLama Factory的对话模板

# 4.vllm推理模型时自定义对话模板

# 案例：使用vllm有效部署Lora微调后的Qwen模型